# Analysis

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

## Data Loading

In [ ]:
# Load processed dataframe
df = pd.read_parquet("../data/processed/listings_processed.parquet")
df.head(3)

## Linear Regression Model

### Initial Feature Transformations

We will often construct a plot with a lowess-fit, following is a sample code:

In [ ]:
def lowess_plot(data, col, colname):
    sns.regplot(data=data, x=col, y="log_price", lowess=True,
                scatter_kws={"alpha": 0.2}, line_kws={"color": "red"})
    plt.xlabel(f"{colname}")
    plt.ylabel("Log price")
    plt.title(f"Log price vs. {colname} (LOWESS)")

#### Accommodates

The relationship appears linear, therefore `accommodates` is not transformed.

In [ ]:
# Make scatter plot with lowess-curve
lowess_plot(df, "accommodates", "Number of guests")
plt.show()

#### Bedrooms

The variable `bedrooms` is treated categorically, because the relation to the price is non-linear and accommodations with 0 bedrooms are on average more expensive than ones with 1 bedroom. 
Listings with more than 6 bedrooms are capped at 5, since only 25 listings exceed this value, which are too few for a stable estimate. 

In [ ]:
count = (df["bedrooms"]>5).sum()
print(f"{count} listings have more than 5 bedrooms.")

In [ ]:
# Create scatterplot of mean log price by bedrooms
mean = df.groupby("bedrooms")["log_price"].agg(["mean"]).reset_index()
sns.scatterplot(data=mean, x="bedrooms", y="mean")
plt.xlabel("Number of bedrooms")
plt.ylabel("Mean log price")
plt.title("Mean price per number of bedrooms")
plt.show()

# Create categories by number of bedrooms
df["bedrooms_cat"] = np.where(df["bedrooms"] > 5, "6+", df["bedrooms"].astype(str))
df["bedrooms_cat"] = pd.Categorical(
    df["bedrooms_cat"], categories=["0", "1", "2", "3", "4", "5", "6+"], ordered=True
)

#### Number_of_reviews_ltm

The relationship appears linear, therefore `number_of_reviews_ltm` is not transformed.

In [ ]:
# Make scatter plot with lowess-curve
lowess_plot(df, "number_of_reviews_ltm", "Number of reviews (last 12 months)")
plt.show()

#### Avg_rating

The log price starts to visibly increase above an `avg_rating` of 4.85 (orange line). To capture this, a linear spline term is added to allow for a different slope above the knot.

In [ ]:
# Make scatter plot with lowess-curve
filtered = df[df["avg_rating"]>0]
lowess_plot(filtered, "avg_rating", "Average rating")

# Add vertical line
knot = 4.85 # Knot, read from the lowess-curve
plt.axvline(knot, color = "orange")
plt.show()

# Add linear spline term to the dataframe
df["rating_above_knot"] = (df["avg_rating"] > knot) * (df["avg_rating"] - knot)

The piecewise linear fit (orange) now follows the LOWESS curve (red) closely above the knot.
The slight non-linearity at low ratings is ignored, since it is only caused by a few data points.

In [ ]:
# Mini-fit with linear spline
df_model = df.copy()
df_model = df_model[df_model["avg_rating"]>0]
Xr = sm.add_constant(df_model[["avg_rating", "rating_above_knot"]])

# Cast to numpy-float
Xr = Xr.astype("float64")
y = df_model["log_price"].astype("float64")

# Fit the linear regression model
mr = sm.OLS(y, Xr).fit()

# Predictions to plot
grid = np.linspace(df_model["avg_rating"].min(), df_model["avg_rating"].max(), 200)
df_pred = pd.DataFrame({"avg_rating": grid,
                   "rating_above_knot": (grid > knot) * (grid - knot)})
df_pred = sm.add_constant(df_pred)

# Make scatter plot with lowess-curve and model predictions
fig, ax = plt.subplots(figsize=(7, 5))
sns.lineplot(x=grid, y=mr.predict(df_pred), ax=ax,
             label=f"piecewise linear (knot={knot})",
             color="orange", linestyle="--")
lowess_plot(df_model, "avg_rating", "Average rating")
plt.legend()
plt.show()

### Model Building

##### Assumptions
The assumptions of linear regression are...

... for valid coefficient estimates:
- **Linear relation between predictors and target.** Checked after the regression on residual plots of the target and feature variables. Predictors may be transformed before used in the model and a first transformation of features was done based on bivariate plots.
- **No strong multicollinearity.** Checked after the regression with VIF (Variance Inflation Factor) of predictors.
- **No unobserved confounders.** This assumption cannot be checked directly from the data and therefore I will make associative statements in this research rather than causal ones, as they don't need this assumption to hold.

... for valid standard errors:
- **Conditional independence of observations given the predictors.** As discussed in the logistic regression during the missing values handling in the preprocessing, two dependencies can occur in the case of Airbnb listings: Firstly, multiple listings per host - this is handled with cluster-robust standard errors using `host_id`. Secondly, the local dependency of Airbnb listings, which is largely captured by the `neighbourhood` predictor, and checked after the regression with a geographic residual plot and Moran's I test.
- **Homoscedasticity of errors.** Checked after the regression on residual plots of the target and feature variables.
- **Normal distribution of residuals.** Checked after the regression on a QQ-plot.

### Checking Assumptions

## Interpretation